# 24 · 上下文补全：Contextual / Parent-Child / Sentence Window

> 切分再聪明，单个 chunk 也常常缺上下文。比如某个 chunk 只写“采用单线程模型”，读者不知道这在讲 Redis。本课用三种技术把上下文接回去。

**本文件覆盖知识点**：Contextual Retrieval / Contextual Chunk·Embedding·BM25 / Parent-Child Retrieval / Sentence Window Retrieval

In [1]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Contextual Retrieval（Anthropic 提出的思路）

普通 chunk 只有正文；Contextual 让 LLM 为每个 chunk 补一段“说明它处于什么上下文”的前缀：

```text
原 chunk:  Redis采用单线程模型。
加语境后: 在Redis服务器架构中，Redis主要采用单线程事件循环模型处理命令请求。
          （Redis采用单线程模型。）
```

存库与检索都用“补全后的文本”，能让“它指什么”这类问题直接命中。

> 可以再进一步：Contextual Embedding（向量化时带上上下文）与 Contextual BM25（BM25 索引时也带上下文）。

In [2]:
# Contextual 前缀生成示意：给每个 chunk 生成一句“文档内语境”
from dotenv import load_dotenv; load_dotenv()
import os
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def add_context(doc_title, chunk, whole_doc_tail):
    """示意：用规则把标题/前文摘要拼成语境（生产用 LLM 生成）"""
    return f'在「{doc_title}」中，围绕{whole_doc_tail}，本段提到：{chunk}'

print(add_context('Redis 深度剖析', 'Redis采用单线程模型。', 'Redis 服务器架构'))
print('\n检索时拿这个补全后的文本去 embedding / BM25，就能回答“那个单线程是什么”。')

在「Redis 深度剖析」中，围绕Redis 服务器架构，本段提到：Redis采用单线程模型。

检索时拿这个补全后的文本去 embedding / BM25，就能回答“那个单线程是什么”。


In [3]:
# 知识点·真调说明：Contextual Retrieval —— 让 LLM 真为“裸 chunk”补一句文档内语境前缀
doc_title = 'Redis 深度剖析'
doc_scope = '服务器架构：线程与 IO 模型'
chunk = '采用单线程事件循环模型。'   # 注意：正文本身不含“Redis”，读起来不知道在讲谁
out = _llm_live(
    prompt='文档：%s\n本 chunk 所属章节：%s\nchunk 正文：%s\n\n请用不超过 60 字的一句话给这段正文补一个'
           '“语境前缀”，说明它讲的是哪个系统、什么背景，让脱离全文的读者也能看懂“单线程事件循环模型”指什么。只输出前缀本身。'
           % (doc_title, doc_scope, chunk),
    system='你是严谨的技术文档编辑。前缀只能依据上面给定的文档与章节信息补背景，禁止臆造细节。',
    fallback='未配置 Key 的固定样例：\n'
             '在 Redis 服务器架构中，命令的执行主要采用单线程事件循环模型。',
    temperature=0.2,
)
if out is None:
    out = '在 Redis 服务器架构中，命令的执行主要采用单线程事件循环模型。'
    print('（以上为固定样例；下面用样例演示“补全后入库文本”）')
print('补全后入库文本（语境前缀 + 原 chunk）:')
print('  %s（%s）' % (out, chunk))
print('→ 存库 / 检索都用这段“补全文本”，embedding 与 BM25 因而能命中“它指什么”这类问题。')

—— 模型实时输出 ——
Redis 服务器在处理客户端请求时，基于 epoll/kqueue 等 IO 多路复用机制
补全后入库文本（语境前缀 + 原 chunk）:
  Redis 服务器在处理客户端请求时，基于 epoll/kqueue 等 IO 多路复用机制（采用单线程事件循环模型。）
→ 存库 / 检索都用这段“补全文本”，embedding 与 BM25 因而能命中“它指什么”这类问题。


In [ ]:
# 知识点·真调说明：Contextual Chunk —— 同一裸 chunk，有/无语境前缀，模型能否对上“指代”
print('① 只有裸 chunk —— 资料没交代是谁的“单线程模型”，模型无从指认')
_llm_live(
    prompt='根据下面资料回答：这段里的“单线程事件循环模型”是哪个系统 / 组件的？\n\n资料：采用单线程事件循环模型。',
    system='你是严谨的客服，只依据给定资料回答；资料信息不足就如实说明无法确定，不要用自己记忆里的系统去猜。',
    fallback='未配置 Key 的固定样例：\n'
             '资料没有说明这是哪个系统、哪个组件——只有一句“采用单线程事件循环模型”，无法确定所指。',
    temperature=0.2,
)
print()
print('② 带 LLM 补的语境前缀 —— 读者一看就懂，模型能给出确切所指')
_llm_live(
    prompt='根据下面资料回答：这段里的“单线程事件循环模型”是哪个系统 / 组件的？\n\n资料：在 Redis 服务器架构中，'
           '命令执行主要采用单线程事件循环模型。（采用单线程事件循环模型。）',
    system='你是严谨的客服，只依据给定资料回答。',
    fallback='未配置 Key 的固定样例：\n'
             '指 Redis 的命令执行引擎——资料说明在 Redis 服务器架构中，命令处理采用单线程事件循环模型。',
    temperature=0.2,
)
print('同一句裸 chunk，有没有“语境前缀”，决定了模型能否把指代对上。')
print('→ 这就是 Contextual Chunk 的意义：给碎片补上身份，检索与生成才不会被“缺上下文”卡住。')

## 2. Parent-Child Retrieval（父子文档）

孩子小（好召回），父母大（够上下文）。

```text
Parent(父): Redis 章节(整节)
  ├─ Child: Redis单线程     ← 检索只对 Child 向量做
  ├─ Child: Redis IO        ← 命中孩子
  └─ Child: Redis事件循环        │
                          ▼
            把命中的 Child 映射回 Parent，喂给 LLM 整节上下文
```

- **为什么好**：小块向量语义集中、易命中；大块给足背景，生成更稳。
- 现在长上下文模型流行，这条更划算，反正 Parent 塞得下。

## 3. Sentence Window Retrieval（句窗）

向量库只存当前句，命中后把“前后 N 句”一起取回当上下文。

```text
向量库:  [句1][句2][句3]...[句7]...
命中句3 → 取 句1+句2+[句3]+句4+句5  → 上下文
```

- 比“整块入库”更省向量空间、语义更聚焦；
- 命中句在长文档里时，句窗让局部逻辑链保持完整。

## 小结

- Contextual：给碎片补上它所在的上下文；
- Parent-Child：小召回 + 大上下文；
- Sentence Window：细粒度索引 + 邻居补齐。

三者的共性：检索的粒度不等于喂给 LLM 的粒度。